In [1]:
import dlt
import pyarrow.parquet as pq
import fsspec
import toml
import requests
import tempfile
import os
import time
import pandas as pd

secrets_path = "./.dlt/secrets.toml"
secrets = toml.load(secrets_path)

def read_parquet_in_batches(parquet_url, max_retries=5, delay=5, batch_size=50000):
    
    'Descarga parquet remoto con retries y lo lee en batches.'
    
    attempt = 0
    while attempt < max_retries:
        try:
            # Crear archivo temporal
            with tempfile.NamedTemporaryFile(delete=False, suffix=".parquet") as tmp_file:
                print(f"Descargando {parquet_url} ...")
                with requests.get(parquet_url, stream=True, timeout=60) as r:
                    r.raise_for_status()
                    for chunk in r.iter_content(chunk_size=8192):
                        tmp_file.write(chunk)
                tmp_file_path = tmp_file.name

            # Abrir parquet con pyarrow
            parquet_file = pq.ParquetFile(tmp_file_path)
            print('Extrayendo parquet')
            print(f"Total filas: {parquet_file.metadata.num_rows}")
            print(f"Row groups: {parquet_file.num_row_groups}")

            # Iterar batches
            for i, batch in enumerate(parquet_file.iter_batches(batch_size=batch_size)):
                print(f"Procesando batch {i + 1}...")
                df = batch.to_pandas().convert_dtypes()
                print(f"Batch {i + 1} con {len(df)} filas")
                yield df
            break

        except Exception as e:
            attempt += 1
            print(f"Error al descargar o leer parquet (intento {attempt}/{max_retries}): {e}")
            if attempt < max_retries:
                print(f"Reintentando en {delay} segundos...")
                time.sleep(delay)
            else:
                print("Se alcanzó el máximo de reintentos. Abortando.")
                raise
        finally:
            if 'tmp_file_path' in locals() and os.path.exists(tmp_file_path):
                os.remove(tmp_file_path)

@dlt.resource(table_name="Comments/2023")
def comments_2023():
    url = "https://datasets-documentation.s3.eu-west-3.amazonaws.com/stackoverflow/parquet/comments/2023.parquet"
    yield from read_parquet_in_batches(url)

@dlt.resource(table_name="Comments/2024")
def comments_2024():
    url = "https://datasets-documentation.s3.eu-west-3.amazonaws.com/stackoverflow/parquet/comments/2024.parquet"
    yield from read_parquet_in_batches(url)
    
pipeline = dlt.pipeline(
    pipeline_name="Bronze_MinIO_1",
    destination="filesystem",
    dataset_name="Bronze"  # carpeta raíz
)

load_info = pipeline.run(
    [comments_2023, comments_2024],
    loader_file_format="parquet",
    write_disposition="replace"
)

print(load_info)

Descargando https://datasets-documentation.s3.eu-west-3.amazonaws.com/stackoverflow/parquet/comments/2023.parquet ...
Extrayendo parquet
Total filas: 2950672
Row groups: 3
Procesando batch 1...
Batch 1 con 50000 filas
Descargando https://datasets-documentation.s3.eu-west-3.amazonaws.com/stackoverflow/parquet/comments/2024.parquet ...
Extrayendo parquet
Total filas: 580920
Row groups: 1
Procesando batch 1...
Batch 1 con 50000 filas
Procesando batch 2...
Batch 2 con 50000 filas
Procesando batch 2...
Batch 2 con 50000 filas
Procesando batch 3...
Batch 3 con 50000 filas
Procesando batch 3...
Batch 3 con 50000 filas
Procesando batch 4...
Batch 4 con 50000 filas
Procesando batch 4...
Batch 4 con 50000 filas
Procesando batch 5...
Batch 5 con 50000 filas
Procesando batch 5...
Batch 5 con 50000 filas
Procesando batch 6...
Batch 6 con 50000 filas
Procesando batch 6...
Batch 6 con 50000 filas
Procesando batch 7...
Batch 7 con 50000 filas
Procesando batch 7...
Batch 7 con 50000 filas
Procesando ba

In [2]:
@dlt.resource(table_name="Posts/2023")
def posts_2023():
    url = "https://datasets-documentation.s3.eu-west-3.amazonaws.com/stackoverflow/parquet/posts/2023.parquet"
    yield from read_parquet_in_batches(url)

@dlt.resource(table_name="Posts/2024")
def posts_2024():
    url = "https://datasets-documentation.s3.eu-west-3.amazonaws.com/stackoverflow/parquet/posts/2024.parquet"
    yield from read_parquet_in_batches(url)

pipeline = dlt.pipeline(
    pipeline_name="Bronze_MinIO_2",
    destination="filesystem",
    dataset_name="Bronze"  # carpeta raíz
)

load_info = pipeline.run(
    [posts_2023, posts_2024],
    loader_file_format="parquet",
    write_disposition="replace"
)

print(load_info)

Descargando https://datasets-documentation.s3.eu-west-3.amazonaws.com/stackoverflow/parquet/posts/2023.parquet ...
Extrayendo parquet
Total filas: 2001772
Row groups: 7
Procesando batch 1...
Batch 1 con 50000 filas
Descargando https://datasets-documentation.s3.eu-west-3.amazonaws.com/stackoverflow/parquet/posts/2024.parquet ...
Extrayendo parquet
Total filas: 385533
Row groups: 2
Procesando batch 1...
Batch 1 con 50000 filas
Procesando batch 2...
Batch 2 con 50000 filas
Procesando batch 2...
Batch 2 con 50000 filas
Procesando batch 3...
Batch 3 con 50000 filas
Procesando batch 3...
Batch 3 con 50000 filas
Procesando batch 4...
Batch 4 con 50000 filas
Procesando batch 4...
Batch 4 con 50000 filas
Procesando batch 5...
Batch 5 con 50000 filas
Procesando batch 5...
Batch 5 con 50000 filas
Procesando batch 6...
Batch 6 con 50000 filas
Procesando batch 6...
Batch 6 con 50000 filas
Procesando batch 7...
Batch 7 con 50000 filas
Procesando batch 7...
Batch 7 con 50000 filas
Procesando batch 8.

In [3]:
@dlt.resource(table_name="Users")
def users():
    url = "https://datasets-documentation.s3.eu-west-3.amazonaws.com/stackoverflow/parquet/users.parquet"
    yield from read_parquet_in_batches(url)

pipeline = dlt.pipeline(
    pipeline_name="Bronze_MinIO_3",
    destination="filesystem",
    dataset_name="Bronze"  # carpeta raíz
)

load_info = pipeline.run(
    [users],
    loader_file_format="parquet",
    write_disposition="replace"
)

print(load_info)

Descargando https://datasets-documentation.s3.eu-west-3.amazonaws.com/stackoverflow/parquet/users.parquet ...
Extrayendo parquet
Total filas: 22484235
Row groups: 27
Procesando batch 1...
Batch 1 con 50000 filas
Procesando batch 2...
Batch 2 con 50000 filas
Procesando batch 3...
Batch 3 con 50000 filas
Procesando batch 4...
Batch 4 con 50000 filas
Procesando batch 5...
Batch 5 con 50000 filas
Procesando batch 6...
Batch 6 con 50000 filas
Procesando batch 7...
Batch 7 con 50000 filas
Procesando batch 8...
Batch 8 con 50000 filas
Procesando batch 9...
Batch 9 con 50000 filas
Procesando batch 10...
Batch 10 con 50000 filas
Procesando batch 11...
Batch 11 con 50000 filas
Procesando batch 12...
Batch 12 con 50000 filas
Procesando batch 13...
Batch 13 con 50000 filas
Procesando batch 14...
Batch 14 con 50000 filas
Procesando batch 15...
Batch 15 con 50000 filas
Procesando batch 16...
Batch 16 con 50000 filas
Procesando batch 17...
Batch 17 con 50000 filas
Procesando batch 18...
Batch 18 con 